# Host SLM Router v7.1 trên Colab

Deploy `hanoi-router-qwen3-4b-v7-1-gguf` (Q4_K_M, ~2.5GB) qua Ollama + cloudflared tunnel.

**Yêu cầu**: Runtime → GPU (T4 free đủ). Chạy lần lượt; cell tunnel cuối in URL để config `OLLAMA_BASE_URL` ở local.

## 1. Cài Ollama

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!/usr/local/bin/ollama --version

## 2. Cài cloudflared (tunnel public, không cần auth)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 3. Start Ollama server

In [ ]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
if '/usr/local/bin' not in os.environ['PATH']:
    os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']

subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=os.environ.copy(),
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!curl -s http://localhost:11434/api/tags

## 4. Tạo model `hanoi-weather-router` từ GGUF v7.1

In [ ]:
from huggingface_hub import hf_hub_download

prompt_path = hf_hub_download(
    repo_id='daredevil467/hanoi-weather-router-data-v7',
    filename='system_prompt_v7_1.txt',
    repo_type='dataset',
)
SYSTEM_PROMPT_V71 = open(prompt_path, encoding='utf-8').read()
print(f'Loaded prompt: {len(SYSTEM_PROMPT_V71)} chars')

In [ ]:
modelfile = f'''FROM hf.co/daredevil467/hanoi-router-qwen3-4b-v7-1-gguf

TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"

# R18 best-practice: Qwen3 thinking ON requires non-greedy sampling.
# Qwen team (Qwen3-4B model card): "DO NOT use greedy decoding, leads to
# performance degradation and endless repetitions." T=0.6, top_p=0.95,
# top_k=20, min_p=0, repeat_penalty=1.1 (Ollama equivalent of presence_penalty).
PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20
PARAMETER min_p 0
PARAMETER repeat_penalty 1.1
PARAMETER num_predict 1024

SYSTEM """{SYSTEM_PROMPT_V71}"""
'''
open('/content/Modelfile', 'w', encoding='utf-8').write(modelfile)
print(f'Modelfile: {len(modelfile)} chars')

In [ ]:
!ollama create hanoi-weather-router -f /content/Modelfile
!ollama list

## 5. Smoke test (single-turn + multi-turn)

In [ ]:
import requests

# R18: best-practice options cho Qwen3 thinking ON (match Modelfile + slm_router.py).
# Pre-R18 dùng temperature=0.0 → Ollama runtime override sẽ shadow Modelfile, dẫn
# tới greedy decoding + thinking = endless repetition (Qwen team warning).
_OPTS = {
    "temperature": 0.6, "top_p": 0.95, "top_k": 20, "min_p": 0,
    "repeat_penalty": 1.1, "num_predict": 1024,
}

# Single-turn
r = requests.post('http://localhost:11434/api/chat', json={
    'model': 'hanoi-weather-router',
    'messages': [{'role': 'user', 'content': 'Thời tiết Cầu Giấy hôm nay thế nào?'}],
    'stream': False, 'keep_alive': '30m',
    'options': _OPTS,
}, timeout=300)
print('Single-turn:', r.json()['message']['content'])

# Multi-turn — kiểm time anchor inheritance (turn 2 'thứ 3' phải thừa kế 'tuần sau' từ turn 1)
r = requests.post('http://localhost:11434/api/chat', json={
    'model': 'hanoi-weather-router',
    'messages': [
        {'role': 'user', 'content': 'Thời tiết phường Cầu Giấy thứ 2 tuần sau'},
        {'role': 'assistant', 'content': '{"intent":"daily_forecast","scope":"ward","confidence":0.92,"rewritten_query":"Thời tiết Phường Cầu Giấy thứ 2 tuần sau"}'},
        {'role': 'user', 'content': 'thứ 3 thì sao?'},
    ],
    'stream': False, 'keep_alive': '30m',
    'options': _OPTS,
}, timeout=300)
print('Multi-turn:', r.json()['message']['content'])

## 6. Start cloudflared tunnel

In [ ]:
import re, subprocess, threading, time

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
pattern = re.compile(r'https://[\w-]+\.trycloudflare\.com')
start = time.time()
while time.time() - start < 30 and public_url is None:
    line = proc.stdout.readline()
    if not line: continue
    m = pattern.search(line)
    if m: public_url = m.group(0)

threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

if not public_url:
    raise RuntimeError('Không tìm được public URL')

print(f'PUBLIC URL: {public_url}')
print(f'\n.env local:')
print(f'OLLAMA_BASE_URL={public_url}')
print('OLLAMA_MODEL=hanoi-weather-router')
print('USE_SLM_ROUTER=true')

## 7. Test public URL

In [ ]:
import requests
# R18 best-practice options (match cell-12 và slm_router.py)
r = requests.post(f'{public_url}/api/chat', json={
    'model': 'hanoi-weather-router',
    'messages': [{'role': 'user', 'content': 'Ngày mai Đống Đa có mưa không?'}],
    'stream': False, 'keep_alive': '30m',
    'options': {
        "temperature": 0.6, "top_p": 0.95, "top_k": 20, "min_p": 0,
        "repeat_penalty": 1.1, "num_predict": 1024,
    },
}, timeout=120)
print(r.status_code, r.json().get('message', {}).get('content'))

## Giữ runtime sống

In [ ]:
import time
try:
    while True:
        time.sleep(60)
        print('.', end='', flush=True)
except KeyboardInterrupt:
    print('\nStopped.')